# Heart Attack Risk Dataset (China): Baseline Analysis to Recommendations

**In short:** 12.15% of the 239,266 patients had a heart attack. No risk factor, access or socioeconomic group, or combination of them, differs from that rate by more than chance, and models trained on the data perform like random guessing. The dataset looks synthetic, so the policy and priority recommendations come from published evidence, not from patterns in the file. The detailed findings are in section 8.

## 1. Loading the data

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats   # only used for the statistical tests

df = pd.read_csv("heart_attack_china.csv", keep_default_na=False)   # keep "None" in Education_Level as a class
df["y"] = (df["Heart_Attack"] == "Yes").astype(int)

# age bands, used in a few charts
df["Age_Band"] = pd.cut(df["Age"], [29, 39, 49, 59, 69, 79, 89], labels=["30-39", "40-49", "50-59", "60-69", "70-79", "80-89"])

# every categorical column except the outcome, id and the two geography labels that have many values
factors = [c for c in df.select_dtypes(exclude="number").columns if c not in ["Patient_ID", "Heart_Attack", "Province", "Age_Band"]]
print(df.shape, "| factors:", len(factors))

(239266, 30) | factors: 22


## 2. Baseline analysis

First question: how many patients had a heart attack? Every group we look at later is compared against this number.

In [2]:
n = len(df)
yes = df["y"].sum()
p = yes / n
se = np.sqrt(p * (1 - p) / n)
print(f"Heart attack: {yes:,} of {n:,} patients = {p:.2%}")
print(f"95% confidence interval: {p - 1.96*se:.2%} to {p + 1.96*se:.2%}")

fig = px.pie(df, names="Heart_Attack", hole=0.5, title="Heart attack in the dataset")
fig.show()

Heart attack: 29,071 of 239,266 patients = 12.15%
95% confidence interval: 12.02% to 12.28%


About 1 in 8 patients (12.15%) had a heart attack. The confidence interval is very narrow (about 12.0% to 12.3%) because there are over 239,000 rows. The data is imbalanced: someone who guesses "No" for everyone is right 88% of the time, so accuracy is not a useful measure here. Instead we compare **group rates with the baseline**.

In [3]:
rate = lambda col: df.groupby(col, observed=True)["y"].mean().mul(100).reset_index()

fig = px.bar(rate("Age_Band"), x="Age_Band", y="y", labels={"y": "Heart attack rate (%)"}, title="Heart attack rate by age band")
fig.add_hline(y=p * 100, line_dash="dash", annotation_text="baseline")
fig.show()

fig = px.bar(rate("Gender"), x="Gender", y="y", labels={"y": "Heart attack rate (%)"}, title="Heart attack rate by gender")
fig.add_hline(y=p * 100, line_dash="dash", annotation_text="baseline")
fig.show()

In real data the rate climbs steeply with age and is higher in men. Here every age band and both genders sit on the baseline line, so age and gender tell us nothing about who has a heart attack.

Pick any factor from the dropdown to see each of its groups against the baseline. Red bars are above the baseline, blue bars are below it.

In [4]:
fig = go.Figure()
cols = ["Age_Band"] + factors
for i, col in enumerate(cols):
    t = df.groupby(col, observed=True)["y"].agg(["sum", "count"]).reset_index()
    t["rate"] = t["sum"] / t["count"] * 100
    t["diff"] = t["rate"] - p * 100
    fig.add_trace(go.Bar(
        x=t[col].astype(str), y=t["rate"], visible=(i == 0),
        marker_color=["crimson" if d > 0 else "steelblue" for d in t["diff"]],
        customdata=t[["count", "diff"]],
        hovertemplate="%{x}<br>Rate: %{y:.2f}%<br>Patients: %{customdata[0]:,}<br>vs baseline: %{customdata[1]:+.2f} pts<extra></extra>"))

buttons = [dict(label=col, method="update",
                args=[{"visible": [j == i for j in range(len(cols))]}, {"xaxis.title.text": col}])
           for i, col in enumerate(cols)]

fig.add_hline(y=p * 100, line_dash="dash", annotation_text=f"baseline {p:.2%}")
fig.update_layout(updatemenus=[dict(buttons=buttons, x=0, y=1.15, xanchor="left", yanchor="top")],
                  title="Heart attack rate by group vs baseline", xaxis_title="Age_Band",
                  yaxis_title="Heart attack rate (%)", yaxis_range=[0, 16], height=500, showlegend=False)
fig.show()

Almost every bar sits within a fraction of a point of the dashed baseline, so the red and blue colours only show tiny differences. Click through the factors: none has a group that is clearly above the baseline.

Before testing risk factors, two quick checks on whether the data behaves like real patients. These support the statements in the limitations.

In [5]:
bp = df.groupby("Hypertension")["Blood_Pressure"].agg(mean="mean", share_140_plus=lambda s: (s >= 140).mean() * 100).round(1).reset_index()
print(bp.to_string(index=False))
px.bar(bp.melt(id_vars="Hypertension", var_name="measure", value_name="value"), x="Hypertension", y="value", color="measure", barmode="group",
       facet_col="measure", title="Measured blood pressure by hypertension flag", height=350).update_yaxes(matches=None).show()

province_region = pd.crosstab(df["Province"], df["Region"], normalize="index") * 100
px.imshow(province_region, text_auto=".0f", color_continuous_scale="Viridis", title="Share of each province's patients in each region (%)").show()

Hypertension  mean  share_140_plus
          No 134.4            44.4
         Yes 134.7            44.6


Patients flagged as hypertensive have the same average blood pressure and the same share at or above 140 mmHg as the others, so the flag is not linked to the measurement. Each province is also spread about 20% across all five regions, so `Region` is not a grouping of provinces (a real province belongs to one region). Real patient data would show both relationships.

## 3. Risk factor hypothesis testing

Here every group of every factor is tested on its own: for each group (for example Male, Female, Smoker, Non-Smoker, age 30-39) we compare its heart attack rate with the rest of the patients using a chi-square test. The null hypothesis is that the group makes no difference. A p-value below 0.05 would mean the gap is unlikely to be chance. Because we run many tests (one per group), some will fall below 0.05 by luck, so we also apply the Bonferroni correction (0.05 divided by the number of tests).

In [6]:
rows = []
for c in ["Age_Band"] + factors:
    for level in df[c].dropna().unique():
        inside = df[c] == level
        table = [[df.loc[inside, "y"].sum(), (inside & (df["y"] == 0)).sum()],
                 [df.loc[~inside, "y"].sum(), (~inside & (df["y"] == 0)).sum()]]
        pval = stats.chi2_contingency(table)[1]
        rate_in, rate_out = df.loc[inside, "y"].mean() * 100, df.loc[~inside, "y"].mean() * 100
        rows.append({"factor": c, "group": str(level), "patients": int(inside.sum()), "rate %": rate_in,
                     "vs baseline (pts)": rate_in - p * 100, "p-value": pval})

groups_test = pd.DataFrame(rows).sort_values("p-value").reset_index(drop=True)
bonf = 0.05 / len(groups_test)
groups_test["p<0.05"] = groups_test["p-value"] < 0.05
groups_test["after Bonferroni"] = groups_test["p-value"] < bonf
print("Groups tested:", len(groups_test), "| Bonferroni threshold:", round(bonf, 5))
print("Groups with p<0.05:", groups_test["p<0.05"].sum(), "| after Bonferroni:", groups_test["after Bonferroni"].sum())
groups_test.head(10).round(4)

Groups tested: 64 | Bonferroni threshold: 0.00078
Groups with p<0.05: 2 | after Bonferroni: 0


,factor,group,patients,rate %,vs baseline (pts),p-value,p<0.05,after Bonferroni
0,Employment_Status,Retired,79872,12.4061,0.2560,0.0068,True,False
1,Employment_Status,Unemployed,79756,11.9276,-0.2224,0.0189,True,False
2,Cholesterol_Level,Low,79555,11.9766,-0.1735,0.0678,False,False
3,Income_Level,Middle,79320,12.3159,0.1659,0.0815,False,False
4,Cholesterol_Level,High,79620,12.3022,0.1521,0.1092,False,False
5,Age_Band,50-59,39559,11.9265,-0.2236,0.1385,False,False
6,Stress_Level,High,79857,12.0265,-0.1236,0.1926,False,False
7,Obesity,Yes,71745,12.2712,0.1212,0.2379,False,False
8,Obesity,No,167521,12.0982,-0.0519,0.2379,False,False
9,Stress_Level,Low,80120,12.2591,0.1090,0.2495,False,False


In [7]:
fig = px.strip(groups_test, x="factor", y="p-value", color="p<0.05", log_y=True, hover_data=["group", "rate %", "patients"],
               color_discrete_map={True: "crimson", False: "steelblue"}, title="p-value of every group vs the rest (log scale)", height=550)
fig.add_hline(y=0.05, line_dash="dash", annotation_text="0.05")
fig.add_hline(y=bonf, line_dash="dot", annotation_text="Bonferroni")
fig.update_xaxes(tickangle=45, title=None)
fig.show()

Each dot is one group. Dots above the 0.05 line mean no evidence of a difference, and nearly all dots are there. Two dots (the Retired and Unemployed groups of `Employment_Status`) fall below 0.05, which is about what 1 test in 20 does by chance, and none falls below the Bonferroni line. Hover over a dot to see which group it is. Smoking, hypertension, diabetes, previous heart attack and family history, which are strong risk factors in reality, show nothing here.

In [8]:
binary = [c for c in factors if df[c].nunique() == 2 and c != "Gender" and c != "Rural_or_Urban"]
out = []
for c in binary:
    top = sorted(df[c].unique())[-1]          # "Yes" or "Smoker"
    a, b = df[df[c] == top]["y"], df[df[c] != top]["y"]
    rr = a.mean() / b.mean()
    se_log = np.sqrt((1 - a.mean()) / a.sum() + (1 - b.mean()) / b.sum())
    out.append({"factor": f"{c} ({top})", "rr": rr, "low": rr * np.exp(-1.96 * se_log), "high": rr * np.exp(1.96 * se_log)})
rr = pd.DataFrame(out).sort_values("rr")

fig = go.Figure(go.Scatter(x=rr["rr"], y=rr["factor"], mode="markers",
                           error_x=dict(type="data", symmetric=False, array=rr["high"] - rr["rr"], arrayminus=rr["rr"] - rr["low"])))
fig.add_vline(x=1, line_dash="dash")
fig.update_layout(title="Risk ratio: exposed vs not exposed (1.0 = no difference)", xaxis_title="Risk ratio (95% CI)", height=500)
fig.show()

A risk ratio of 1 means the exposed group has the same risk as the unexposed group; 2 would mean double. Every factor has a ratio of about 1.0 and every interval crosses 1. For comparison, studies of real populations find ratios well above 1 for smoking, diabetes and previous heart attack.

### Numeric variables

Age, blood pressure and the CVD risk score are numbers, so they are compared between patients with and without a heart attack using Welch's t-test (compares two averages). Cohen's d is the size of the gap in standard deviations (0.2 would be small).

In [9]:
rows = []
for c in ["Age", "Blood_Pressure", "CVD_Risk_Score"]:
    a, b = df.loc[df["y"] == 1, c], df.loc[df["y"] == 0, c]
    t, pval = stats.ttest_ind(a, b, equal_var=False)
    d = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
    rows.append({"variable": c, "mean (heart attack)": a.mean(), "mean (no heart attack)": b.mean(), "difference": a.mean() - b.mean(), "p-value": pval, "Cohen's d": d})
display(pd.DataFrame(rows).round(3))

sample = df.sample(30000, random_state=1)
px.box(sample.melt(id_vars="Heart_Attack", value_vars=["Age", "Blood_Pressure", "CVD_Risk_Score"]), x="Heart_Attack", y="value", color="Heart_Attack",
       facet_col="variable", title="Age, blood pressure and risk score by outcome (30,000-patient sample)", height=400).update_yaxes(matches=None).show()

,variable,mean (heart attack),mean (no heart attack),difference,p-value,Cohen's d
0,Age,59.410,59.474,-0.064,0.557,-0.004
1,Blood_Pressure,134.551,134.508,0.043,0.792,0.002
2,CVD_Risk_Score,54.313,54.509,-0.197,0.227,-0.008


All three variables have almost identical averages in both outcome groups and Cohen's d is about 0 (the largest gap is the risk score, about 0.2 points on a 10-99 scale). The CVD risk score, which is meant to rise with risk, is no higher in patients who had a heart attack. The boxes in the chart are the same height and position.

### Combinations of risk factors

Single factors can look harmless while a combination is risky (for example an older smoker with hypertension). Here smoking, hypertension and age band are combined into 24 groups, and each is tested against all other patients (chi-square, Bonferroni correction).

In [10]:
g = df.groupby(["Age_Band", "Smoking_Status", "Hypertension"], observed=True)["y"].agg(["sum", "count"]).reset_index()
pvals = []
for _, r in g.iterrows():
    rest_y, rest_n = df["y"].sum() - r["sum"], len(df) - r["count"]
    pvals.append(stats.chi2_contingency([[r["sum"], r["count"] - r["sum"]], [rest_y, rest_n - rest_y]])[1])
g["rate"] = g["sum"] / g["count"] * 100
g["p"] = pvals
g["group"] = g["Smoking_Status"] + " / hypertension " + g["Hypertension"]
print("Groups:", len(g), "| p<0.05:", (g["p"] < 0.05).sum(), "| after Bonferroni:", (g["p"] < 0.05 / len(g)).sum())
print(f"Rates range from {g['rate'].min():.2f}% to {g['rate'].max():.2f}% (baseline {p*100:.2f}%)")
px.imshow(g.pivot(index="Age_Band", columns="group", values="rate"), text_auto=".1f", color_continuous_scale="Turbo",
          title="Heart attack rate (%) by age band, smoking and hypertension").show()

Groups: 24 | p<0.05: 2 | after Bonferroni: 0
Rates range from 11.54% to 12.81% (baseline 12.15%)


The highest and lowest of the 24 combinations are within about a point of the baseline, and none is significant after the Bonferroni correction (the count above is about what chance gives). Even the combination that should be worst in reality, an older smoker with hypertension, is at the baseline. So combinations of risk factors show nothing either.

## 4. Healthcare access and socioeconomic inequality

The brief asks whether people with poorer access or lower socioeconomic standing carry more of the burden. First, who has poor access?

In [11]:
fig = px.histogram(df, x="Rural_or_Urban", color="Healthcare_Access", barnorm="percent", text_auto=".1f",
                   category_orders={"Healthcare_Access": ["Poor", "Moderate", "Good"]},
                   title="Healthcare access by rural / urban (%)")
fig.show()

fig = px.histogram(df, x="Income_Level", color="Healthcare_Access", barnorm="percent", text_auto=".1f",
                   category_orders={"Income_Level": ["Low", "Middle", "High"], "Healthcare_Access": ["Poor", "Moderate", "Good"]},
                   title="Healthcare access by income level (%)")
fig.show()

In reality, rural and low-income patients have clearly worse access. Here the split is the same everywhere (about a third Poor, Moderate, Good), so the dataset does not show an access gap between rural and urban or between rich and poor.

In [12]:
groups = ["Healthcare_Access", "Hospital_Availability", "Income_Level", "Education_Level", "Employment_Status", "Rural_or_Urban"]
rows = []
for g in groups:
    t = df.groupby(g)["y"].agg(["sum", "count"])
    t["rate"] = t["sum"] / t["count"] * 100
    t["err"] = 1.96 * np.sqrt(t["rate"] / 100 * (1 - t["rate"] / 100) / t["count"]) * 100
    for level, r in t.iterrows():
        rows.append({"group": g, "level": level, "rate": r["rate"], "err": r["err"]})
ineq = pd.DataFrame(rows)

fig = px.bar(ineq, x="level", y="rate", error_y="err", facet_col="group", facet_col_wrap=3,
             labels={"rate": "Heart attack rate (%)"}, title="Heart attack rate by access and socioeconomic group (95% CI)", height=650)
fig.update_xaxes(matches=None, showticklabels=True, title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.add_hline(y=p * 100, line_dash="dash")
fig.show()

All groups have rates between about 11.9% and 12.4% and every error bar overlaps the baseline. Poor access, low income, no education and living in a rural area do not go with a higher heart attack rate here.

In [13]:
heat = df.pivot_table(index="Income_Level", columns="Healthcare_Access", values="y", aggfunc="mean").mul(100)
heat = heat.loc[["Low", "Middle", "High"], ["Poor", "Moderate", "Good"]]
fig = px.imshow(heat, text_auto=".2f", color_continuous_scale="Turbo", zmin=heat.values.min(), zmax=heat.values.max(),
                labels=dict(color="Rate (%)"), title="Heart attack rate (%): income vs healthcare access (colour scale zoomed in)")
fig.show()
print(f"Lowest cell {heat.values.min():.2f}%, highest cell {heat.values.max():.2f}%, baseline {p*100:.2f}%")

Lowest cell 11.81%, highest cell 12.48%, baseline 12.15%


The colour scale is zoomed to the range actually present in the data, so even tiny gaps get different colours. Read the numbers, not just the colours: the lowest and highest cells are only a fraction of a percentage point apart. Cells with about 26,000 patients vary by around 0.4 points by chance alone, so this spread is noise.

## 5. Policy review

The dataset has no policy variables, so this section compares the dataset with published national figures and summarises what China's programmes have achieved. Sources are listed at the end.

In [14]:
compare = pd.DataFrame({
    "Measure": ["Hypertension (adults)", "Smokers (adults)", "Diabetes (adults)", "Obesity (adults)"],
    "This dataset": [f"{(df['Hypertension']=='Yes').mean():.1%}", f"{(df['Smoking_Status']=='Smoker').mean():.1%}",
                     f"{(df['Diabetes']=='Yes').mean():.1%}", f"{(df['Obesity']=='Yes').mean():.1%}"],
    "National figure": ["31.6% (2020-22)", "26.6% (age 15+)", "11.2% (2015-17, WHO standard)", "17.8% (2020-22)"],
})
compare

,Measure,This dataset,National figure
0,Hypertension (adults),27.0%,31.6% (2020-22)
1,Smokers (adults),50.0%,26.6% (age 15+)
2,Diabetes (adults),11.1%,"11.2% (2015-17, WHO standard)"
3,Obesity (adults),30.0%,17.8% (2020-22)


Diabetes is close to the national figure. Smoking (about double) and obesity (30% vs 17.8%) are well above it and hypertension is below it. So the file does not look like China's population.

Next, a pattern that is well documented in real data: hypertension rises steeply with age. The 2020 CHARLS survey (19,282 adults aged 45 and over) gives the rate by age group.

In [15]:
charls = pd.DataFrame({"age": ["45-54", "55-64", "65-74", "75-84", "85-94"], "CHARLS 2020 (%)": [20.78, 32.41, 43.47, 53.26, 54.13]})
ours = df.assign(a=pd.cut(df["Age"], [44, 54, 64, 74, 84, 94], labels=charls["age"])).dropna(subset=["a"])
charls["This dataset (%)"] = ours.groupby("a", observed=True)["Hypertension"].apply(lambda s: (s == "Yes").mean() * 100).values

fig = go.Figure()
for col in ["CHARLS 2020 (%)", "This dataset (%)"]:
    fig.add_trace(go.Scatter(x=charls["age"], y=charls[col], mode="lines+markers", name=col))
fig.update_layout(title="Hypertension rate by age group: real survey vs this dataset", xaxis_title="Age group", yaxis_title="Hypertension (%)", yaxis_range=[0, 60])
fig.show()

In the real survey the rate nearly triples from 20.8% to 54.1% between the 45-54 and 85-94 groups. In this dataset it is a flat line at about 27%. Another CHARLS finding: the hypertension rate is almost the same in city centres (37.49%) and villages (37.78%), so a gap between rural and urban areas is not expected for hypertension itself.

| Policy / evidence | What it does | Numbers |
|---|---|---|
| Basic Public Health Service (from 2009) | Free hypertension and diabetes check-ups and follow-up for people aged 35+ at community level | 2014 pilot: hypertension control 76.9% vs 68.2% in comparison areas |
| Chest pain centres | Faster treatment for heart attack through accredited hospitals | Odds of in-hospital death 0.67 (95% CI 0.51 to 0.88), in 798,008 patients across 746 hospitals (2013-2016) |
| Air Pollution Action Plan (APPCAP) | National cut in PM2.5 | PM2.5 fell from 52.5 to 42.2 µg/m³; about 64,000 (6.8%) fewer pollution-related deaths |
| Tobacco tax (May 2015) | Wholesale ad valorem tax raised from 5% to 11% and a new ¥0.10 per pack | Average retail price up 11% (¥11.6 to ¥12.8 a pack); cigarette sales down 7.8% (127 to 117 billion packs); modelled estimate of 2.2 to 6.5 million fewer smokers |
| Healthy China 2030 | National health plan with a tobacco target | Smoking from 26.6% (2018) to below 20% by 2030; the source also calls for a comprehensive smoke-free law and further tax rises |
| Volume-based drug procurement (2019, 11 pilot cities) | Bulk buying of medicines, including 7 blood pressure drugs | Average price cut 59.1% for the 7 antihypertensives (52% across 25 drugs). In Shenzhen, volume of these drugs rose 92% and spending fell 49% (one city, purchase data) |
| Primary-care blood pressure monitoring (CHARLS 2020) | Community clinics measure blood pressure of hypertensive patients | 72.6% receive it, about 11.9 times a year; fewer than 50% get health education on complications |

**The scale of the problem (2023 national cardiovascular report):** 330 million people have cardiovascular disease, 245 million have hypertension and 11.39 million have coronary heart disease. Hospital costs for cardiovascular disease (excluding stroke) were 212.11 billion yuan in 2022, 43.9% of it for coronary heart disease. People aged 60 and over account for 64.9% of these costs.

**Gaps that remain:** only 43.3% of adults with hypertension know they have it, 38.7% are treated and 12.9% have it under control (2020-22). In the CHARLS sample only 27.25% of hypertensive patients were found through a routine check-up. Coronary heart disease mortality in 2021 was higher in rural (148.19 per 100,000) than urban (135.08 per 100,000) areas.

## 6. Predictive modelling

The brief asks for predictive insights, so we train models. Because only 12% of patients had a heart attack, **accuracy is not used** (always guessing "No" already scores 88%). We use **precision** (of the patients the model flags, how many really had a heart attack) and **recall** (of all real heart attacks, how many the model finds), plus the precision-recall AUC. Random guessing gets a precision of 12.15%, so that is the number to beat.

Two models are trained, both with settings that limit overfitting (shallow trees, a minimum number of patients per leaf, regularisation, subsampling). Overfitting is checked by comparing train and test scores.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier

X = pd.get_dummies(df.drop(columns=["Patient_ID", "Heart_Attack", "y"]).assign(Age_Band=lambda d: d["Age_Band"].astype(str)), drop_first=True)
y = df["y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Features:", X.shape[1], "| train rows:", len(X_train), "| test rows:", len(X_test))

models = {
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=200, class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=50,
                             reg_lambda=10, reg_alpha=1, scale_pos_weight=(1 - y.mean()) / y.mean(), n_jobs=2, random_state=42),
}

Features: 51 | train rows: 191412 | test rows: 47854


In [17]:
rows, curves = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    for part, Xp, yp in [("train", X_train, y_train), ("test", X_test, y_test)]:
        prob = model.predict_proba(Xp)[:, 1]
        pred = prob > 0.5
        rows.append({"model": name, "data": part, "precision": precision_score(yp, pred), "recall": recall_score(yp, pred),
                     "F1": f1_score(yp, pred), "PR-AUC": average_precision_score(yp, prob), "ROC-AUC": roc_auc_score(yp, prob)})
        if part == "test":
            curves[name] = precision_recall_curve(yp, prob)

rows.append({"model": "Random guessing", "data": "test", "precision": y_test.mean(), "recall": 1.0,
             "F1": 2 * y_test.mean() / (1 + y_test.mean()), "PR-AUC": y_test.mean(), "ROC-AUC": 0.5})
pd.DataFrame(rows).round(3)

,model,data,precision,recall,F1,PR-AUC,ROC-AUC
0,Random forest,train,0.157,0.566,0.246,0.171,0.604
1,Random forest,test,0.123,0.444,0.192,0.123,0.503
2,XGBoost,train,0.142,0.535,0.225,0.148,0.565
3,XGBoost,test,0.121,0.454,0.190,0.121,0.500
4,Random guessing,test,0.121,1.000,0.217,0.121,0.500


In [18]:
fig = go.Figure()
for name, (prec, rec, _) in curves.items():
    fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines", name=name))
fig.add_hline(y=y_test.mean(), line_dash="dash", annotation_text="random guessing")
fig.update_layout(title="Precision-recall curve on the test set", xaxis_title="Recall", yaxis_title="Precision", yaxis_range=[0, 0.5], height=450)
fig.show()

Read the table like this: on the test set both models have a precision of about 12-13%, which is the same as random guessing (12.15%), and their PR-AUC and ROC-AUC (about 0.50) are no better than chance. The random forest scores better on the training data (ROC-AUC 0.60) than on the test data (0.50). That gap is overfitting: it memorised random patterns in the training rows that do not repeat in new patients, even with the regularisation settings. XGBoost overfits less (0.565 vs 0.50). The recall of 45% looks decent but comes with a precision at the baseline, so the models are just flagging many patients at random. The precision-recall curves lie on the random-guessing line.

This is not because the categories are evenly split. A model needs a *link between the inputs and the outcome*, and the earlier tests already showed that no factor has one. The models confirm it from a different angle.

In [19]:
imp = pd.Series(models["XGBoost"].feature_importances_, index=X.columns).sort_values().tail(10)
px.bar(imp, orientation="h", labels={"value": "Importance", "index": ""}, title="XGBoost: top 10 feature importances (all small and close together)").update_layout(showlegend=False).show()

The importances are spread almost evenly across features, with no standout variable. When a model has found a real signal, one or two features dominate. Here it is just fitting noise.

### Other checks that tell us whether a model can work

A model's score only means something if it beats what the same model gets on data with no signal. Two further checks:

1. **Shuffled-label test.** Shuffle the outcome so it is random by construction, train the same XGBoost model, and see what PR-AUC it gets. If the real model is not clearly above the shuffled ones, it found nothing.
2. **Regularised logistic regression.** A simple model with a penalty. If even the simple linear view finds nothing, a more complex model is not hiding something simple.

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

real = average_precision_score(y_test, models["XGBoost"].predict_proba(X_test)[:, 1])
rng = np.random.default_rng(0)
shuffled = []
for _ in range(5):
    m = XGBClassifier(**models["XGBoost"].get_params()).fit(X_train, rng.permutation(y_train.values))
    shuffled.append(average_precision_score(y_test, m.predict_proba(X_test)[:, 1]))

logit = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, class_weight="balanced", max_iter=500)).fit(X_train, y_train)
logit_prob = logit.predict_proba(X_test)[:, 1]

print(f"XGBoost PR-AUC (real labels):      {real:.4f}")
print(f"XGBoost PR-AUC (shuffled labels):  {np.mean(shuffled):.4f}  (range {min(shuffled):.4f} to {max(shuffled):.4f})")
print(f"Logistic regression PR-AUC / ROC-AUC: {average_precision_score(y_test, logit_prob):.4f} / {roc_auc_score(y_test, logit_prob):.4f}")
print(f"Random guessing PR-AUC:            {y_test.mean():.4f}")

XGBoost PR-AUC (real labels):      0.1213
XGBoost PR-AUC (shuffled labels):  0.1223  (range 0.1205 to 0.1232)
Logistic regression PR-AUC / ROC-AUC: 0.1220 / 0.4984
Random guessing PR-AUC:            0.1215


The model trained on real labels scores the same as the models trained on random labels, and the logistic regression is also at the random-guessing level. So the models are not finding anything.

**Why a useful model cannot be trained on this dataset**

- **No input is related to the outcome.** The group tests, risk ratios, AUCs and models all agree. A model can only learn from a link between inputs and outcome, and here there is none.
- **Balanced categories are not the problem on their own.** Evenly split inputs would be fine if they were related to the outcome. The issue is the missing relationship.
- **The 12% class imbalance is manageable** (class weights, precision and recall), and the models handle it. It is not the cause either.
- **The best possible model here is "12.15% for everyone".** Any higher-risk label it assigns to a group is chance.
- **What would help:** real clinical data, where age, blood pressure, smoking and diabetes are related to the outcome, dates and follow-up (so the outcome is a future event), and the policy start-year sheet. The same code would then be reused.

### Fairness across groups

A model should perform the same way for every group. Here the XGBoost model is scored separately for each gender, area, income level and region on the test set. A group where the score is clearly different from the others, or from 0.5, would point to bias.

In [21]:
prob = models["XGBoost"].predict_proba(X_test)[:, 1]
info = df.loc[X_test.index].reset_index(drop=True)
yt = y_test.reset_index(drop=True)
rows = []
for c in ["Gender", "Rural_or_Urban", "Income_Level", "Region"]:
    for level in info[c].unique():
        m = (info[c] == level).values
        rows.append({"group": c, "level": level, "patients": int(m.sum()), "real rate %": yt[m].mean() * 100,
                     "ROC-AUC": roc_auc_score(yt[m], prob[m]), "precision": precision_score(yt[m], prob[m] > 0.5), "recall": recall_score(yt[m], prob[m] > 0.5)})
fair = pd.DataFrame(rows)
display(fair.round(3))
fig = px.scatter(fair, x="level", y="ROC-AUC", facet_col="group", facet_col_wrap=4, color="group", range_y=[0.45, 0.55], title="XGBoost ROC-AUC by group (0.5 = random guessing)", height=400)
fig.update_xaxes(matches=None, showticklabels=True, title=None)
fig.add_hline(y=0.5, line_dash="dash")
fig.show()

,group,level,patients,real rate %,ROC-AUC,precision,recall
0,Gender,Female,24083,12.353,0.506,0.124,0.453
1,Gender,Male,23771,11.943,0.494,0.117,0.456
2,Rural_or_Urban,Urban,19083,12.309,0.502,0.122,0.479
3,Rural_or_Urban,Rural,28771,12.043,0.499,0.120,0.438
4,Income_Level,Middle,15807,12.349,0.501,0.123,0.543
5,Income_Level,Low,16055,12.040,0.490,0.115,0.375
6,Income_Level,High,15992,12.062,0.508,0.122,0.444
7,Region,Western,9677,11.967,0.499,0.117,0.516
8,Region,Southern,9597,12.368,0.496,0.118,0.406
9,Region,Eastern,9496,12.068,0.500,0.119,0.535


The model scores about 0.50 in every group, with only small random differences between them, and its precision stays at each group's own heart attack rate. Recall varies more (about 38% to 54% across groups) because each group is small and the model flags patients at random. So it is not biased against any gender, area, income level or region, but only because it performs like random guessing for all of them. This would need to be rechecked on real data, where a model can be better for one group than another.

## 7. Strategic recommendations

Because the dataset shows no real associations, it cannot tell us which groups or regions to prioritise. These recommendations, including who to prioritise, come from the external evidence in section 5, not from patterns in the file.

**Who to prioritise (external evidence):** adults aged 65 and over (hypertension in the CHARLS survey rises from 20.8% at 45-54 to 54.1% at 85-94), rural residents (higher coronary heart disease mortality), and people with undiagnosed or uncontrolled hypertension (only 43.3% aware, 12.9% controlled).

1. **Close the hypertension care gap.** Awareness (43.3%), treatment (38.7%) and control (12.9%) are the weakest links. Expand blood pressure screening at village clinics and community centres, especially for the rural population.
2. **Target rural areas first.** Coronary heart disease mortality in 2021 was higher in rural areas (148.19 per 100,000) than urban (135.08), per the 2023 national cardiovascular report, so give rural hospitals chest pain centre status and ambulance links to speed up heart attack treatment.
3. **Keep cutting air pollution.** APPCAP shows that pollution policy saves lives. Continue it, with extra attention to regions that are still above the national average.
4. **Keep reducing smoking.** 26.6% of adults smoke (over 300 million people), so tobacco taxes and smoke-free rules still have a large room to improve.
5. **Collect better data.** Add dates and follow-up, true geography, facility-level access data and a second worksheet that records which policy or law started in which year, so patients can be split into before and after each measure. The layout could look like this: `Policy | Start year | Region covered | What it does`. For example, the Basic Public Health Service started in 2009. With that sheet, the heart attack rate before and after each policy could be compared properly.
6. **Re-run the models on better data.** The same pipeline (precision, recall, PR-AUC, regularised trees) can be reused once real risk factors are in the data.

## 8. Findings

What the analysis showed, section by section:

- **Baseline:** 12.15% of the 239,266 patients (29,071) had a heart attack, about 1 in 8.
- **Risk factors:** none of the groups tested (every level of every factor) is meaningfully linked to heart attack. The few p-values below 0.05 are what chance produces and none survives the Bonferroni correction. Risk ratios are all about 1.0.
- **Combinations and numbers:** the 24 smoker x hypertension x age-band combinations and the three numeric variables (age, blood pressure, risk score) show nothing either.
- **Inequality:** poor access, low income, low education, unemployment and rural residence do not carry a higher heart attack rate. Access is also spread equally between rural and urban and between income groups.
- **Policy:** real-world evidence says hypertension control, chest pain centres and air quality policy matter, but this dataset cannot be used to evaluate them.
- **Prediction:** a random forest and an XGBoost model (regularised, judged by precision and recall, not accuracy) perform like random guessing on the test set (the random forest even overfits, scoring 0.60 ROC-AUC on training data and 0.50 on test data). The shuffled-label test and logistic regression agree, and the scores are the same across gender, area, income and region.

**Overall:** the dataset is clean but behaves like random data, probably synthetic. The honest conclusion for the brief is that it cannot answer questions about risk factors or inequality, and the recommendations rest on external evidence.

## 9. Limitations

- The data looks synthetic: flat distributions, round percentages, and no relationships between variables (hypertension does not match blood pressure, provinces are not inside regions).
- There are no dates or follow-up, so 12.15% is a share of patients in the file and not a yearly incidence.
- `Region`, `Province` and access are labels without real geography, so no regional comparison is meaningful.
- Cross-sectional data cannot show cause and effect.
- Many tests were run, so a lone small p-value should not be trusted without correction.
- National figures in section 5 come from different years and populations (CHARLS covers only adults aged 45 and over, and its management figures are self-reported), so they are for context only.

**Sources used in section 5**

- Report on Cardiovascular Health and Diseases in China 2023 (Biomedical and Environmental Sciences): 330 million CVD, 245 million hypertension, hypertension 31.6% / 43.3% / 38.7% / 12.9%, smoking 26.6%, diabetes 11.2%, obesity 17.8%, CHD mortality 2021, hospital costs 2022.
- Zhu A, et al. Prevalence and management of hypertension among adults aged 45 and older in mainland China: insights from the 2020 CHARLS survey. Medicine 2026;105(8):e47704 (19,282 participants; hypertension by age, residence and management).
- Basic Public Health Service and hypertension control: PLoS ONE (2021), role of the Basic Public Health Service program in hypertension control in China.
- Chest pain centre accreditation and acute coronary syndrome outcomes: BMJ Quality & Safety (2020).
- Air Pollution Action Plan (2013) and avoided deaths: Nature Communications (2020).
- Early assessment of China's 2015 tobacco tax increase (price, sales and tax figures).
- Tobacco control and Healthy China 2030 (smoking target 26.6% to below 20%).
- Volume-based procurement and antihypertensive drugs in Shenzhen (price cut and volume figures).